#  ⭐ `fat_medicamentos`


In [1]:
%run ../../../config/bootstrap.py

import pandas as pd
import numpy as np
from pathlib import Path
from utils import get_project_root, normalize_date_column, normalize_rows, apply_mappings, fuzzy_merge##, normalize_duration, expandir_gravidade_wide
project_root = get_project_root() 

# 🥉 Bronze 

Raw Data
as is production, low quality

In [2]:
path = Path(project_root) / "data/01_bronze/anvisa/Medicamentos/Medicamentos.parquet"
bronze = pd.read_parquet(path) 
bronze.columns

Index(['IDENTIFICACAO_NOTIFICACAO', 'RELACAO_MEDICAMENTO_EVENTO',
       'NOME_MEDICAMENTO_WHODRUG', 'PRINCIPIOS_ATIVOS_WHODRUG', 'CODIGO_ATC',
       'DETENTOR_REGISTRO', 'CONCENTRACAO', 'COMPONENTE_SUSPEITO',
       'ACAO_ADOTADA', 'PROBLEMAS_ADICIONAIS_RELCIONADOS_MEDICAMENTO',
       'INDICACAO_MEDDRA', 'INDICACAO_RELATADA_NOTIFICADOR_INICIAL', 'DOSE',
       'FREQUENCIA_DOSE', 'POSOLOGIA', 'DURACAO', 'INICIO_ADMINISTRACAO',
       'FIM_ADMINISTRACAO', 'FORMA_FARMACEUTICA', 'VIA_ADMINISTRACAO',
       'VIA_ADMINISTRACAO_MAE_PAI', 'NUMELO_LOTE', 'ATUALIZADO'],
      dtype='object')

In [3]:
bronze.head(2)

,IDENTIFICACAO_NOTIFICACAO,RELACAO_MEDICAMENTO_EVENTO,NOME_MEDICAMENTO_WHODRUG,PRINCIPIOS_ATIVOS_WHODRUG,CODIGO_ATC,DETENTOR_REGISTRO,CONCENTRACAO,COMPONENTE_SUSPEITO,ACAO_ADOTADA,PROBLEMAS_ADICIONAIS_RELCIONADOS_MEDICAMENTO,INDICACAO_MEDDRA,INDICACAO_RELATADA_NOTIFICADOR_INICIAL,DOSE,FREQUENCIA_DOSE,POSOLOGIA,DURACAO,INICIO_ADMINISTRACAO,FIM_ADMINISTRACAO,FORMA_FARMACEUTICA,VIA_ADMINISTRACAO,VIA_ADMINISTRACAO_MAE_PAI,NUMELO_LOTE,ATUALIZADO
0,BR-ANVISA-300000004,Suspeito,Oxacilina,Oxacillin sodium,J01CF,Teuto,None,None,None,None,None,None,250 milligram (mg),6 hora,None,4 dia,20181124,None,None,infusão intravenosa,None,5833018,2026-01-15
1,BR-ANVISA-300000005,Suspeito,Paracemol,Paracetamol,N02BE,None,None,None,Retirada do medicamento,None,None,None,None,None,None,None,20181122,20181122,None,oral,None,None,2026-01-15


# 🥈 Silver

normalized data, medium quality


In [4]:
silver = bronze.copy()
silver.shape

(591457, 23)

## ✅ hash

In [5]:
from utils import build_row_hash

silver["HASH_BRONZE"] = build_row_hash(
    silver, 
    silver.columns.difference(['ATUALIZADO']).tolist(), 
    algo="sha256", 
    sep="|"
)

## ✅ Missing

In [6]:
silver.isna().sum()

IDENTIFICACAO_NOTIFICACAO                            0
RELACAO_MEDICAMENTO_EVENTO                        1033
NOME_MEDICAMENTO_WHODRUG                          5313
PRINCIPIOS_ATIVOS_WHODRUG                         5843
CODIGO_ATC                                        5313
DETENTOR_REGISTRO                               452994
CONCENTRACAO                                    491185
COMPONENTE_SUSPEITO                             530567
ACAO_ADOTADA                                    285907
PROBLEMAS_ADICIONAIS_RELCIONADOS_MEDICAMENTO    571291
INDICACAO_MEDDRA                                346290
INDICACAO_RELATADA_NOTIFICADOR_INICIAL          339119
DOSE                                            362716
FREQUENCIA_DOSE                                   3583
POSOLOGIA                                       291599
DURACAO                                         504788
INICIO_ADMINISTRACAO                            309631
FIM_ADMINISTRACAO                               422856
FORMA_FARM

In [7]:
hist_silver = silver.copy()
for col in silver.select_dtypes(include=['object']).columns:
    silver[col] = normalize_rows(silver[col])

In [8]:
hist_silver.isna().sum()

IDENTIFICACAO_NOTIFICACAO                            0
RELACAO_MEDICAMENTO_EVENTO                        1033
NOME_MEDICAMENTO_WHODRUG                          5313
PRINCIPIOS_ATIVOS_WHODRUG                         5843
CODIGO_ATC                                        5313
DETENTOR_REGISTRO                               452994
CONCENTRACAO                                    491185
COMPONENTE_SUSPEITO                             530567
ACAO_ADOTADA                                    285907
PROBLEMAS_ADICIONAIS_RELCIONADOS_MEDICAMENTO    571291
INDICACAO_MEDDRA                                346290
INDICACAO_RELATADA_NOTIFICADOR_INICIAL          339119
DOSE                                            362716
FREQUENCIA_DOSE                                   3583
POSOLOGIA                                       291599
DURACAO                                         504788
INICIO_ADMINISTRACAO                            309631
FIM_ADMINISTRACAO                               422856
FORMA_FARM

In [9]:
hist_silver.shape

(591457, 24)

## ✅ DATAS

In [10]:
colunas_data = ["INICIO_ADMINISTRACAO", "FIM_ADMINISTRACAO"]

In [11]:
hist_silver[colunas_data].info()
hist_silver[colunas_data].head()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 591457 entries, 0 to 591456
Data columns (total 2 columns):
 #   Column                Non-Null Count   Dtype 
---  ------                --------------   ----- 
 0   INICIO_ADMINISTRACAO  281826 non-null  object
 1   FIM_ADMINISTRACAO     168601 non-null  object
dtypes: object(2)
memory usage: 9.0+ MB


,INICIO_ADMINISTRACAO,FIM_ADMINISTRACAO
0,20181124,None
1,20181122,20181122
2,20181103,20181115
3,20181016,20181115
4,20181024,None


In [12]:
for col in colunas_data:
    if col in hist_silver.columns:
        hist_silver[col] = normalize_date_column(hist_silver[col])

In [13]:
hist_silver[colunas_data].info()
hist_silver[colunas_data].head()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 591457 entries, 0 to 591456
Data columns (total 2 columns):
 #   Column                Non-Null Count   Dtype         
---  ------                --------------   -----         
 0   INICIO_ADMINISTRACAO  235655 non-null  datetime64[ns]
 1   FIM_ADMINISTRACAO     153422 non-null  datetime64[ns]
dtypes: datetime64[ns](2)
memory usage: 9.0 MB


,INICIO_ADMINISTRACAO,FIM_ADMINISTRACAO
0,2018-11-24,NaT
1,2018-11-22,2018-11-22
2,2018-11-03,2018-11-15
3,2018-10-16,2018-11-15
4,2018-10-24,NaT


In [14]:
hist_silver.shape

(591457, 24)

## ✅ Mappings

- RELACAO_MED_EVENTO
- COMPONENTE_SUSPEITO
- ACAO_ADOTADA

In [15]:
# RELACAO_MEDICAMENTO_EVENTO
relacao_medicamento_evento_map = {
    "SUSPEITO": 1,
    "CONCOMITANTE": 2,
    "MEDICAMENTO NAO ADMINISTRADO": 3,
    "INTERACAO": 4,
}
# COMPONENTE_SUSPEITO
componente_suspeito_map = {
    "PRINCÍPIO ATIVO": 1, 
    "EXCIPIENTE, NÃO CLASSIFICADO": 2, 
    "SOLVENTE": 3, 
    "CORANTE": 4,
    "CONSERVANTE": 5, 
    "AGENTE FLAVORIZADOR": 6, 
    "EXCESSO PERCENTUAL": 7, 
    "ANTIOXIDANTE": 8,
    "ESTABILIZANTE": 9
}

# ACAO_ADOTADA
acao_adotada_map = {
    "RETIRADA DO MEDICAMENTO": 1, 
    "SEM ALTERACAO DA DOSE": 2, 
    "NAO APLICAVEL": 3,
    "REDUCAO DA DOSE": 4, 
    "AUMENTO DA DOSE": 5
}
mappings_config = [
    {
        "kind": "dict",
        "col": "RELACAO_MEDICAMENTO_EVENTO",
        "map": relacao_medicamento_evento_map,
        "fillna_valor": "DESCONHECIDO",
        "fillna_chave": 0,
        "tipo_int64": True,
        "drop_original": False,
    },
    {
        "kind": "dict",
        "col": "COMPONENTE_SUSPEITO",
        "map": componente_suspeito_map,
        "fillna_valor": "DESCONHECIDO",
        "fillna_chave": 0,
        "tipo_int64": True,
        "drop_original": False,
    },
    {
        "kind": "dict",
        "col": "ACAO_ADOTADA",
        "map": acao_adotada_map,
        "fillna_valor": "DESCONHECIDO",
        "fillna_chave": 0,
        "tipo_int64": True,
        "drop_original": False,
    }]

In [16]:
hist_silver = apply_mappings(hist_silver, mappings_config)

In [17]:
hist_silver.shape

(591457, 30)

In [18]:
hist_silver[['ACAO_ADOTADA','ACAO_ADOTADA_VALOR','ACAO_ADOTADA_CHAVE']].value_counts(dropna=False)

ACAO_ADOTADA             ACAO_ADOTADA_VALOR       ACAO_ADOTADA_CHAVE
NaN                      DESCONHECIDO             0                     285907
Retirada do medicamento  RETIRADA DO MEDICAMENTO  1                      92706
Desconhecido             DESCONHECIDO             0                      84500
Sem alteração da dose    SEM ALTERAÇÃO DA DOSE    2                      59566
Não aplicável            NÃO APLICÁVEL            3                      55100
Redução da dose          REDUÇÃO DA DOSE          4                       8882
Aumento da dose          AUMENTO DA DOSE          5                       4796
Name: count, dtype: int64

In [19]:
hist_silver[['COMPONENTE_SUSPEITO','COMPONENTE_SUSPEITO_VALOR','COMPONENTE_SUSPEITO_CHAVE']].value_counts(dropna=False)

COMPONENTE_SUSPEITO           COMPONENTE_SUSPEITO_VALOR     COMPONENTE_SUSPEITO_CHAVE
NaN                           DESCONHECIDO                  0                            530567
Princípio Ativo               PRINCÍPIO ATIVO               1                             60624
Excipiente, não classificado  EXCIPIENTE, NÃO CLASSIFICADO  2                               131
Corante                       CORANTE                       4                                39
Conservante                   CONSERVANTE                   5                                37
Solvente                      SOLVENTE                      3                                25
Excesso percentual            EXCESSO PERCENTUAL            7                                12
Antioxidante                  ANTIOXIDANTE                  8                                 8
Agente Flavorizador           AGENTE FLAVORIZADOR           6                                 7
Estabilizante                 ESTABILIZANTE       

In [20]:
hist_silver[['RELACAO_MEDICAMENTO_EVENTO'
,'RELACAO_MEDICAMENTO_EVENTO_VALOR'
,'RELACAO_MEDICAMENTO_EVENTO_CHAVE']].value_counts(dropna=False)

RELACAO_MEDICAMENTO_EVENTO    RELACAO_MEDICAMENTO_EVENTO_VALOR  RELACAO_MEDICAMENTO_EVENTO_CHAVE
Suspeito                      SUSPEITO                          1                                   401828
Concomitante                  CONCOMITANTE                      2                                   176549
Medicamento não administrado  MEDICAMENTO NÃO ADMINISTRADO      3                                     9852
Interação                     INTERAÇÃO                         4                                     2195
NaN                           DESCONHECIDO                      0                                     1033
Name: count, dtype: int64

## ✅ DURACAO

In [21]:
from utils import normalize_duracao

In [22]:
hist_silver = normalize_duracao(hist_silver, coluna="DURACAO")

In [23]:
hist_silver[['DURACAO','DURACAO_TIPO_CHAVE','DURACAO_TIPO_VALOR','DURACAO_VALOR']].value_counts(dropna=False).head(10)

DURACAO    DURACAO_TIPO_CHAVE  DURACAO_TIPO_VALOR  DURACAO_VALOR
NaN        0                   DESCONHECIDO        NaN              504788
1 dia      4                   DIA                 1.0               22808
2 dia      4                   DIA                 2.0                4437
3 dia      4                   DIA                 3.0                3309
1 hora     3                   HORA                1.0                3255
5 dia      4                   DIA                 5.0                2666
1 minuto   2                   MINUTO              1.0                2450
4 dia      4                   DIA                 4.0                2382
30 minuto  2                   MINUTO              30.0               2355
7 dia      4                   DIA                 7.0                2011
Name: count, dtype: int64

In [24]:
hist_silver.shape

(591457, 33)

## ✅ CONCENTRACAO

In [25]:
from utils.med_normalize_concentracao import normalize_concentracao


In [26]:
hist_silver = normalize_concentracao(hist_silver, col="CONCENTRACAO")

In [27]:
hist_silver.columns

Index(['IDENTIFICACAO_NOTIFICACAO', 'RELACAO_MEDICAMENTO_EVENTO',
       'NOME_MEDICAMENTO_WHODRUG', 'PRINCIPIOS_ATIVOS_WHODRUG', 'CODIGO_ATC',
       'DETENTOR_REGISTRO', 'CONCENTRACAO', 'COMPONENTE_SUSPEITO',
       'ACAO_ADOTADA', 'PROBLEMAS_ADICIONAIS_RELCIONADOS_MEDICAMENTO',
       'INDICACAO_MEDDRA', 'INDICACAO_RELATADA_NOTIFICADOR_INICIAL', 'DOSE',
       'FREQUENCIA_DOSE', 'POSOLOGIA', 'DURACAO', 'INICIO_ADMINISTRACAO',
       'FIM_ADMINISTRACAO', 'FORMA_FARMACEUTICA', 'VIA_ADMINISTRACAO',
       'VIA_ADMINISTRACAO_MAE_PAI', 'NUMELO_LOTE', 'ATUALIZADO', 'HASH_BRONZE',
       'RELACAO_MEDICAMENTO_EVENTO_VALOR', 'RELACAO_MEDICAMENTO_EVENTO_CHAVE',
       'COMPONENTE_SUSPEITO_VALOR', 'COMPONENTE_SUSPEITO_CHAVE',
       'ACAO_ADOTADA_VALOR', 'ACAO_ADOTADA_CHAVE', 'DURACAO_TIPO_VALOR',
       'DURACAO_TIPO_CHAVE', 'DURACAO_VALOR', 'CONCENTRACAO_TIPO_VALOR',
       'CONCENTRACAO_TIPO_CHAVE', 'CONCENTRACAO_VALOR',
       'CONCENTRACAO_VALOR_NUMERADOR', 'CONCENTRACAO_VALOR_DENOMINADOR

In [28]:
hist_silver.query("CONCENTRACAO_TIPO_CHAVE!='desconhecido'")[['CONCENTRACAO',
       'CONCENTRACAO_TIPO_CHAVE',
       'CONCENTRACAO_TIPO_VALOR', 
       'CONCENTRACAO_VALOR',
       'CONCENTRACAO_VALOR_NUMERADOR', 
       'CONCENTRACAO_VALOR_DENOMINADOR']].value_counts(dropna=False).head(10)

CONCENTRACAO  CONCENTRACAO_TIPO_CHAVE  CONCENTRACAO_TIPO_VALOR  CONCENTRACAO_VALOR  CONCENTRACAO_VALOR_NUMERADOR  CONCENTRACAO_VALOR_DENOMINADOR
NaN           0                        desconhecido             NaN                 NaN                           NaN                               491185
100mg         1                        milligram (mg)           100.0               100.0                         NaN                                 2557
500mg         1                        milligram (mg)           500.0               500.0                         NaN                                 2279
1g            2                        gram (g)                 1.0                 1.0                           NaN                                 1989
500 mg        1                        milligram (mg)           500.0               500.0                         NaN                                 1510
100 mg        1                        milligram (mg)           100.0           

In [29]:
hist_silver.CONCENTRACAO_TIPO_VALOR.value_counts(dropna=False)

CONCENTRACAO_TIPO_VALOR
desconhecido                        499050
milligram (mg)                       46849
milligram per millilitre (mg/mL)     31856
gram (g)                              7076
percent (%)                           2610
millilitre (mL)                       2192
microgram (ug)                         740
gram per millilitre (g/mL)             657
milligram per milligram (mg/mg)        362
milligram per gram (mg/g)               65
Name: count, dtype: int64

In [30]:
hist_silver.CONCENTRACAO_TIPO_CHAVE.value_counts(dropna=False)

CONCENTRACAO_TIPO_CHAVE
0    499050
1     46849
6     31856
2      7076
5      2610
4      2192
3       740
9       657
7       362
8        65
Name: count, dtype: Int64

## ✅ DOSE

In [31]:
from utils.med_normalize_dose import normalize_dose

In [32]:
hist_silver = normalize_dose(hist_silver, col="DOSE")

In [33]:
hist_silver.query("DOSE_TIPO_CHAVE!='desconhecido'")[['DOSE',
'DOSE_TIPO_CHAVE', 
'DOSE_TIPO_VALOR', 
'DOSE_VALOR']].value_counts(dropna=False).head(10)

DOSE                  DOSE_TIPO_CHAVE  DOSE_TIPO_VALOR     DOSE_VALOR
NaN                   0                desconhecido        NaN           362716
100 milligram (mg)    1                milligram (mg)      100.0          10888
50 milligram (mg)     1                milligram (mg)      50.0           10206
10 milligram (mg)     1                milligram (mg)      10.0            8807
300 milligram (mg)    1                milligram (mg)      300.0           7805
20 milligram (mg)     1                milligram (mg)      20.0            7264
1 dosage form ({DF})  5                dosage form ({DF})  1.0             7168
40 milligram (mg)     1                milligram (mg)      40.0            6389
500 milligram (mg)    1                milligram (mg)      500.0           6151
1 gram (g)            3                gram (g)            1.0             5719
Name: count, dtype: int64

In [34]:
hist_silver.shape

(591457, 41)

## Fuzzy

### ✅ VIA_ADM

In [35]:
from utils.med_normalize_via_adm import normalizar_via_fuzzy


In [36]:
hist_silver["VIA_ADMINISTRACAO_CHAVE"] = (
    hist_silver["VIA_ADMINISTRACAO"]
    .astype(str)
    .apply(lambda txt: normalizar_via_fuzzy(txt, score_threshold=80)["chave"])
)

hist_silver["VIA_ADMINISTRACAO_MAE_PAI_CHAVE"] = (
    hist_silver["VIA_ADMINISTRACAO_MAE_PAI"]
    .astype(str)
    .apply(lambda txt: normalizar_via_fuzzy(txt, score_threshold=80)["chave"])
)

In [37]:
hist_silver.columns

Index(['IDENTIFICACAO_NOTIFICACAO', 'RELACAO_MEDICAMENTO_EVENTO',
       'NOME_MEDICAMENTO_WHODRUG', 'PRINCIPIOS_ATIVOS_WHODRUG', 'CODIGO_ATC',
       'DETENTOR_REGISTRO', 'CONCENTRACAO', 'COMPONENTE_SUSPEITO',
       'ACAO_ADOTADA', 'PROBLEMAS_ADICIONAIS_RELCIONADOS_MEDICAMENTO',
       'INDICACAO_MEDDRA', 'INDICACAO_RELATADA_NOTIFICADOR_INICIAL', 'DOSE',
       'FREQUENCIA_DOSE', 'POSOLOGIA', 'DURACAO', 'INICIO_ADMINISTRACAO',
       'FIM_ADMINISTRACAO', 'FORMA_FARMACEUTICA', 'VIA_ADMINISTRACAO',
       'VIA_ADMINISTRACAO_MAE_PAI', 'NUMELO_LOTE', 'ATUALIZADO', 'HASH_BRONZE',
       'RELACAO_MEDICAMENTO_EVENTO_VALOR', 'RELACAO_MEDICAMENTO_EVENTO_CHAVE',
       'COMPONENTE_SUSPEITO_VALOR', 'COMPONENTE_SUSPEITO_CHAVE',
       'ACAO_ADOTADA_VALOR', 'ACAO_ADOTADA_CHAVE', 'DURACAO_TIPO_VALOR',
       'DURACAO_TIPO_CHAVE', 'DURACAO_VALOR', 'CONCENTRACAO_TIPO_VALOR',
       'CONCENTRACAO_TIPO_CHAVE', 'CONCENTRACAO_VALOR',
       'CONCENTRACAO_VALOR_NUMERADOR', 'CONCENTRACAO_VALOR_DENOMINADOR

In [38]:
hist_silver[["VIA_ADMINISTRACAO","VIA_ADMINISTRACAO_CHAVE"]].value_counts(dropna=False).head()

VIA_ADMINISTRACAO    VIA_ADMINISTRACAO_CHAVE
NaN                  5                          330673
oral                 5                           31428
infusão intravenosa  6                           26376
intramuscular        6                           19896
desconhecida         0                           18764
Name: count, dtype: int64

In [39]:
hist_silver[["VIA_ADMINISTRACAO_MAE_PAI","VIA_ADMINISTRACAO_MAE_PAI_CHAVE"]].value_counts(dropna=False).head()

VIA_ADMINISTRACAO_MAE_PAI   VIA_ADMINISTRACAO_MAE_PAI_CHAVE
NaN                         5                                  591302
E2B R2 Code: 050 - Other    6                                      45
Oral                        5                                      15
E2B R2 Code: 065 - Unknown  0                                      14
Unknown                     0                                      14
Name: count, dtype: int64

### ✅ DETENTOR_REGISTRO

In [40]:
hist_dim_detentor_registro = pd.read_parquet(Path(project_root) / "data/02_silver/hist_dim_detentor_registro/hist_dim_detentor_registro.parquet")
hist_dim_detentor_registro[['DETENTOR_REGISTRO','DETENTOR_REGISTRO_CHAVE', 'DETENTOR_REGISTRO_VALOR']].drop_duplicates().head()

,DETENTOR_REGISTRO,DETENTOR_REGISTRO_CHAVE,DETENTOR_REGISTRO_VALOR
0,ABACUS MEDICINE,1,ABACUS MEDICINE
1,ABBOLT,2,ABBOT
2,ABBOT,2,ABBOT
3,ABBOT BRASIL,2,ABBOT
4,ABBOT L: 0071/20 V: 31/01/2022,2,ABBOT


In [41]:
hist_silver = fuzzy_merge(
    dim_df = hist_dim_detentor_registro,
    fact_df = hist_silver,
    dim_id_col="DETENTOR_REGISTRO_CHAVE",
    dim_text_col="DETENTOR_REGISTRO",
    fact_text_col="DETENTOR_REGISTRO",
    threshold=95, 
    suffix="" , 
    dedupe_on=False,
)

In [42]:
hist_silver['DETENTOR_REGISTRO_CHAVE'] = hist_silver['DETENTOR_REGISTRO_CHAVE'].fillna(0) 
hist_silver['DETENTOR_REGISTRO_CHAVE'].value_counts(dropna=False).head()

DETENTOR_REGISTRO_CHAVE
0      477515
223     11160
237      9325
83       7976
161      5470
Name: count, dtype: Int64

In [43]:
silver.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 591457 entries, 0 to 591456
Data columns (total 24 columns):
 #   Column                                        Non-Null Count   Dtype 
---  ------                                        --------------   ----- 
 0   IDENTIFICACAO_NOTIFICACAO                     591457 non-null  object
 1   RELACAO_MEDICAMENTO_EVENTO                    590424 non-null  object
 2   NOME_MEDICAMENTO_WHODRUG                      586144 non-null  object
 3   PRINCIPIOS_ATIVOS_WHODRUG                     585614 non-null  object
 4   CODIGO_ATC                                    586144 non-null  object
 5   DETENTOR_REGISTRO                             138463 non-null  object
 6   CONCENTRACAO                                  100272 non-null  object
 7   COMPONENTE_SUSPEITO                           60890 non-null   object
 8   ACAO_ADOTADA                                  305550 non-null  object
 9   PROBLEMAS_ADICIONAIS_RELCIONADOS_MEDICAMENTO  20166 non-nul

### ✅ FORMA_FARMACEUTICA

In [44]:
hist_dim_forma_farmaceutica = pd.read_parquet(Path(project_root) / "data/02_silver/hist_dim_forma_farmaceutica/hist_dim_forma_farmaceutica.parquet")
hist_dim_forma_farmaceutica[['FORMA_FARMACEUTICA','FORMA_FARMACEUTICA_CHAVE', 'FORMA_FARMACEUTICA_VALOR']].drop_duplicates().head()

,FORMA_FARMACEUTICA,FORMA_FARMACEUTICA_CHAVE,FORMA_FARMACEUTICA_VALOR
0,Adesivo Transdérmico,1,Adesivo transdermico
1,adesivo,1,Adesivo transdermico
2,adesivo transdérmico,1,Adesivo transdermico
3,Adesivo,1,Adesivo transdermico
5,ADESIVO,1,Adesivo transdermico


In [45]:
hist_silver.shape

(591457, 45)

In [46]:
hist_silver = fuzzy_merge(
    dim_df = hist_dim_forma_farmaceutica,
    fact_df = hist_silver,
    dim_id_col="FORMA_FARMACEUTICA_CHAVE",
    dim_text_col="FORMA_FARMACEUTICA",
    fact_text_col="FORMA_FARMACEUTICA",
    threshold=95, 
    suffix="" , 
    dedupe_on=False,
)

In [47]:
hist_silver.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 591457 entries, 0 to 591456
Data columns (total 47 columns):
 #   Column                                        Non-Null Count   Dtype         
---  ------                                        --------------   -----         
 0   IDENTIFICACAO_NOTIFICACAO                     591457 non-null  object        
 1   RELACAO_MEDICAMENTO_EVENTO                    590424 non-null  object        
 2   NOME_MEDICAMENTO_WHODRUG                      586144 non-null  object        
 3   PRINCIPIOS_ATIVOS_WHODRUG                     585614 non-null  object        
 4   CODIGO_ATC                                    586144 non-null  object        
 5   DETENTOR_REGISTRO                             113942 non-null  object        
 6   CONCENTRACAO                                  100272 non-null  object        
 7   COMPONENTE_SUSPEITO                           60890 non-null   object        
 8   ACAO_ADOTADA                                  305550 n

In [48]:
hist_silver['FORMA_FARMACEUTICA_CHAVE'] = hist_silver['FORMA_FARMACEUTICA_CHAVE'].fillna(0) 

In [49]:
hist_silver[['FORMA_FARMACEUTICA_CHAVE']].value_counts(dropna=False)

FORMA_FARMACEUTICA_CHAVE
0                           430359
14                           71146
4                            37369
12                           16150
16                           14393
11                           11338
2                             5318
10                            1501
15                            1489
3                              595
1                              345
20                             271
8                              246
19                             184
18                             155
13                             127
9                              122
6                              113
5                               94
7                               88
17                              54
Name: count, dtype: int64

### ✅ INDICACAO_MEDDRA

In [50]:
dim_soc_llt = pd.read_parquet(Path(project_root) / "data/03_gold/dim_soc_llt/dim_soc_llt.parquet").query("PRIMARY_FLAG=='Y'")
dim_soc_llt = dim_soc_llt.rename(columns={"LLT_NAME": "LLT"})
dim_soc_llt.head(2)

,REACAO_CHAVE,PRIMARY_FLAG,SOC_CODE,SOC_NAME,SOC_ABBREV,PRIMARY_SOC,HLGT_CODE,HLGT_NAME,HLT_CODE,HLT_NAME,PT_CODE,PT_NAME,LLT_CODE,LLT
0,SOC_10005329_HLGT_10002086_HLT_10002042_PT_10002043_LLT_10002043,Y,10005329,Distúrbios dos sistemas hematológico e linfático,Blood,10005329,10002086,Anemias não hemolíticas e depressão medular,10002042,Anemiais carenciais,10002043,Anemia por deficiência de folato,10002043,Anemia por deficiência de folato
1,SOC_10005329_HLGT_10002086_HLT_10002042_PT_10002043_LLT_10002044,Y,10005329,Distúrbios dos sistemas hematológico e linfático,Blood,10005329,10002086,Anemias não hemolíticas e depressão medular,10002042,Anemiais carenciais,10002043,Anemia por deficiência de folato,10002044,Anemia por deficiência de ácido fólico


In [51]:
hist_silver = fuzzy_merge(
    dim_df=dim_soc_llt[['REACAO_CHAVE', 'LLT']],
    fact_df=hist_silver,
    dim_id_col="REACAO_CHAVE",
    dim_text_col="LLT",
    fact_text_col="INDICACAO_MEDDRA",
    threshold=98,
    suffix="_x"  # opcional; default "_MATCH"
)

In [52]:
hist_silver.columns

Index(['IDENTIFICACAO_NOTIFICACAO', 'RELACAO_MEDICAMENTO_EVENTO',
       'NOME_MEDICAMENTO_WHODRUG', 'PRINCIPIOS_ATIVOS_WHODRUG', 'CODIGO_ATC',
       'DETENTOR_REGISTRO', 'CONCENTRACAO', 'COMPONENTE_SUSPEITO',
       'ACAO_ADOTADA', 'PROBLEMAS_ADICIONAIS_RELCIONADOS_MEDICAMENTO',
       'INDICACAO_MEDDRA', 'INDICACAO_RELATADA_NOTIFICADOR_INICIAL', 'DOSE',
       'FREQUENCIA_DOSE', 'POSOLOGIA', 'DURACAO', 'INICIO_ADMINISTRACAO',
       'FIM_ADMINISTRACAO', 'FORMA_FARMACEUTICA', 'VIA_ADMINISTRACAO',
       'VIA_ADMINISTRACAO_MAE_PAI', 'NUMELO_LOTE', 'ATUALIZADO', 'HASH_BRONZE',
       'RELACAO_MEDICAMENTO_EVENTO_VALOR', 'RELACAO_MEDICAMENTO_EVENTO_CHAVE',
       'COMPONENTE_SUSPEITO_VALOR', 'COMPONENTE_SUSPEITO_CHAVE',
       'ACAO_ADOTADA_VALOR', 'ACAO_ADOTADA_CHAVE', 'DURACAO_TIPO_VALOR',
       'DURACAO_TIPO_CHAVE', 'DURACAO_VALOR', 'CONCENTRACAO_TIPO_VALOR',
       'CONCENTRACAO_TIPO_CHAVE', 'CONCENTRACAO_VALOR',
       'CONCENTRACAO_VALOR_NUMERADOR', 'CONCENTRACAO_VALOR_DENOMINADOR

### ✅ INDICACAO_RELATADA_NOTIFICADOR_INICIAL

In [53]:
hist_silver.INDICACAO_RELATADA_NOTIFICADOR_INICIAL.value_counts().head()

INDICACAO_RELATADA_NOTIFICADOR_INICIAL
PRODUCT USED FOR UNKNOWN INDICATION    12215
DRUG USE FOR UNKNOWN INDICATION         6457
DOENÇA DE CROHN                         3220
DIABETES                                3157
HIPERTENSÃO                             2975
Name: count, dtype: int64

In [54]:
hist_silver = fuzzy_merge(
    dim_df = dim_soc_llt[['REACAO_CHAVE', 'LLT']],
    fact_df = hist_silver,
    dim_id_col="REACAO_CHAVE",
    dim_text_col="LLT",
    fact_text_col="INDICACAO_RELATADA_NOTIFICADOR_INICIAL",
    threshold = 98,
    suffix="_y")



In [55]:
hist_silver.columns

Index(['IDENTIFICACAO_NOTIFICACAO', 'RELACAO_MEDICAMENTO_EVENTO',
       'NOME_MEDICAMENTO_WHODRUG', 'PRINCIPIOS_ATIVOS_WHODRUG', 'CODIGO_ATC',
       'DETENTOR_REGISTRO', 'CONCENTRACAO', 'COMPONENTE_SUSPEITO',
       'ACAO_ADOTADA', 'PROBLEMAS_ADICIONAIS_RELCIONADOS_MEDICAMENTO',
       'INDICACAO_MEDDRA', 'INDICACAO_RELATADA_NOTIFICADOR_INICIAL', 'DOSE',
       'FREQUENCIA_DOSE', 'POSOLOGIA', 'DURACAO', 'INICIO_ADMINISTRACAO',
       'FIM_ADMINISTRACAO', 'FORMA_FARMACEUTICA', 'VIA_ADMINISTRACAO',
       'VIA_ADMINISTRACAO_MAE_PAI', 'NUMELO_LOTE', 'ATUALIZADO', 'HASH_BRONZE',
       'RELACAO_MEDICAMENTO_EVENTO_VALOR', 'RELACAO_MEDICAMENTO_EVENTO_CHAVE',
       'COMPONENTE_SUSPEITO_VALOR', 'COMPONENTE_SUSPEITO_CHAVE',
       'ACAO_ADOTADA_VALOR', 'ACAO_ADOTADA_CHAVE', 'DURACAO_TIPO_VALOR',
       'DURACAO_TIPO_CHAVE', 'DURACAO_VALOR', 'CONCENTRACAO_TIPO_VALOR',
       'CONCENTRACAO_TIPO_CHAVE', 'CONCENTRACAO_VALOR',
       'CONCENTRACAO_VALOR_NUMERADOR', 'CONCENTRACAO_VALOR_DENOMINADOR

## Pivot

### ✅ PROBLEMAS_ADICIONAIS_RELCIONADOS_MEDICAMENTO

In [56]:
from utils import expandir_lista_wide

In [57]:
hist_silver = expandir_lista_wide(
    hist_silver,
    col="PROBLEMAS_ADICIONAIS_RELCIONADOS_MEDICAMENTO",
    prefix="PROB_ADIC_",   # se quiser um prefixo mais curto/bonito
)


In [58]:

hist_silver.filter(like="PROB_ADIC_").head(2)

,PROB_ADIC_ABUSO,PROB_ADIC_ERRO_DE_MEDICACAO,PROB_ADIC_EXPOSICAO_OCUPACIONAL,PROB_ADIC_FALSIFICACAO,PROB_ADIC_LOTES_TESTADOS_E_DENTRO_DAS_ESPECIFICACOES,PROB_ADIC_LOTES_TESTADOS_E_FORA_DAS_ESPECIFICACOES,PROB_ADIC_MEDICAMENTO_TOMADO_FORA_DA_DATA_DE_VALIDADE,PROB_ADIC_MEDICAMENTO_TOMADO_PELO_PAI,PROB_ADIC_SUPERDOSE,PROB_ADIC_USO_INCORRETO,PROB_ADIC_USO_OFF_LABEL_USO_SEM_REGISTRO
0,0,0,0,0,0,0,0,0,0,0,0
1,0,0,0,0,0,0,0,0,0,0,0


In [59]:
hist_silver.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 591457 entries, 0 to 591456
Data columns (total 64 columns):
 #   Column                                                 Non-Null Count   Dtype         
---  ------                                                 --------------   -----         
 0   IDENTIFICACAO_NOTIFICACAO                              591457 non-null  object        
 1   RELACAO_MEDICAMENTO_EVENTO                             590424 non-null  object        
 2   NOME_MEDICAMENTO_WHODRUG                               586144 non-null  object        
 3   PRINCIPIOS_ATIVOS_WHODRUG                              585614 non-null  object        
 4   CODIGO_ATC                                             586144 non-null  object        
 5   DETENTOR_REGISTRO                                      113942 non-null  object        
 6   CONCENTRACAO                                           100272 non-null  object        
 7   COMPONENTE_SUSPEITO                                    6

### ✅ CODIGO_ATC

In [60]:
from utils import desagrupar_quando_atc_multiplo


In [61]:
print(f"Shape antes da desagregação: {hist_silver.shape}")
# Só explode nome/princípios junto com ATC quando há MULTIPLOS códigos ATC.
# Ex. A03DB|N02BB → sim; Buscopan com um único A03DB e 2 APIs → não.
hist_silver_dedup = desagrupar_quando_atc_multiplo(
    hist_silver.rename(columns={'CODIGO_ATC': 'ATC_CODE_LEVEL_4'}),
    atc_col='ATC_CODE_LEVEL_4',
    colunas_alinhadas=['PRINCIPIOS_ATIVOS_WHODRUG', 'NOME_MEDICAMENTO_WHODRUG'],
)
print(f"Shape depois da desagregação: {hist_silver_dedup.shape}")


Shape antes da desagregação: (591457, 64)
Shape depois da desagregação: (809971, 64)


## Versionamento do `hist_fat_medicamentos`

| VERSION_ID | DATE_UPDATE | Conteúdo |
| --- | --- | --- |
| 1 | 2025-02-24 | Carga da `main` (804 658) — histórica |
| 2 | 2026-08-07 | Desag ATC condicional + `ATC_LEVEL_5` (809 971) — atual |
| 3+ | (auto) | Próximas cargas |

**Para publicar uma v3 (ou vN):**
1. Rode o notebook até a silver (desagregar → ATC_LEVEL_5).
2. Na célula abaixo, deixe `VERSION_ID = None` (auto = max do hist + 1) **ou** force `VERSION_ID = 3`.
3. Rode `publish_snapshot_version` — a versão anterior fica `CURRENT_FLAG=False`; a nova vira atual.
4. Rode a gold (`load_current_version`) — só a versão atual vai para `fat_medicamentos`.

Não apague versões antigas do hist; a gold sempre filtra `CURRENT_FLAG=True`.


In [ ]:
from utils import (
    add_atc_level_5,
    publish_snapshot_version,
    load_current_version,
    stamp_snapshot_version,
)

hist_path = Path(project_root) / "data/02_silver/hist_fat_medicamentos/hist_fat_medicamentos.parquet"

# ATC_LEVEL_5 na SILVER (não na gold)
dim_atc = pd.read_parquet(Path(project_root) / "data/03_gold/dim_atc/dim_atc.parquet")
hist_silver_dedup = add_atc_level_5(hist_silver_dedup, dim_atc)

# ---------------------------------------------------------------------------
# VERSION_ID desta execução
#   None  -> automático (max(VERSION_ID) no hist + 1)  ← use isso na v3, v4, ...
#   int   -> força um id (ex.: 3)
# ---------------------------------------------------------------------------
VERSION_ID = None  # <- na v3 normalmente deixe None (ou VERSION_ID = 3)

if VERSION_ID is None:
    if hist_path.exists():
        _vids = pd.read_parquet(hist_path, columns=["VERSION_ID"])["VERSION_ID"]
        VERSION_ID = int(pd.to_numeric(_vids, errors="coerce").max()) + 1
    else:
        VERSION_ID = 1

hist_silver_dedup = stamp_snapshot_version(hist_silver_dedup, VERSION_ID, current=True)
print(f"Silver rows: {len(hist_silver_dedup):,} | publicando VERSION_ID={VERSION_ID}")

hash_exclude = [
    "ATUALIZADO",
    "HASH_BRONZE",
    "VERSION_ID",
    "CURRENT_FLAG",
    "DATE_UPDATE",
    "ATC_LEVEL_5_SCORE",
]
hist_silver_dedup["HASH_SILVER"] = build_row_hash(
    hist_silver_dedup,
    hist_silver_dedup.columns.difference(hash_exclude).tolist(),
    algo="sha256",
    sep="|",
)


In [63]:
hist_silver_dedup.columns

Index(['IDENTIFICACAO_NOTIFICACAO', 'RELACAO_MEDICAMENTO_EVENTO',
       'NOME_MEDICAMENTO_WHODRUG', 'PRINCIPIOS_ATIVOS_WHODRUG',
       'ATC_CODE_LEVEL_4', 'DETENTOR_REGISTRO', 'CONCENTRACAO',
       'COMPONENTE_SUSPEITO', 'ACAO_ADOTADA',
       'PROBLEMAS_ADICIONAIS_RELCIONADOS_MEDICAMENTO', 'INDICACAO_MEDDRA',
       'INDICACAO_RELATADA_NOTIFICADOR_INICIAL', 'DOSE', 'FREQUENCIA_DOSE',
       'POSOLOGIA', 'DURACAO', 'INICIO_ADMINISTRACAO', 'FIM_ADMINISTRACAO',
       'FORMA_FARMACEUTICA', 'VIA_ADMINISTRACAO', 'VIA_ADMINISTRACAO_MAE_PAI',
       'NUMELO_LOTE', 'ATUALIZADO', 'HASH_BRONZE',
       'RELACAO_MEDICAMENTO_EVENTO_VALOR', 'RELACAO_MEDICAMENTO_EVENTO_CHAVE',
       'COMPONENTE_SUSPEITO_VALOR', 'COMPONENTE_SUSPEITO_CHAVE',
       'ACAO_ADOTADA_VALOR', 'ACAO_ADOTADA_CHAVE', 'DURACAO_TIPO_VALOR',
       'DURACAO_TIPO_CHAVE', 'DURACAO_VALOR', 'CONCENTRACAO_TIPO_VALOR',
       'CONCENTRACAO_TIPO_CHAVE', 'CONCENTRACAO_VALOR',
       'CONCENTRACAO_VALOR_NUMERADOR', 'CONCENTRACAO_VALO

In [ ]:
# Publica no hist: fecha CURRENT_FLAG da versão anterior e grava a nova como atual.
# Republicar o mesmo VERSION_ID substitui só aquela versão (não duplica).
hist_all = publish_snapshot_version(
    hist_silver_dedup,
    hist_path,
    version_id=VERSION_ID,
    archive_unversioned_as=None,
)

print(f"Hist: {hist_path}")
print(hist_all.groupby(["VERSION_ID", "CURRENT_FLAG"], dropna=False).size().rename("rows"))
print(
    f"Atual: VERSION_ID={hist_all.loc[hist_all['CURRENT_FLAG'], 'VERSION_ID'].iloc[0]} "
    f"→ {hist_all['CURRENT_FLAG'].sum():,} linhas"
)


### (One-shot, já feito) Arquivo VERSION_ID=1

A v1 da `main` (804 658, `DATE_UPDATE=2025-02-24`) **já está** no hist. Esta célula só reconfirma.

Para reimportar manualmente (raro): parquet em `/tmp/hist_fat_medicamentos_v1_main.parquet`.


In [ ]:
hist_path = Path(project_root) / "data/02_silver/hist_fat_medicamentos/hist_fat_medicamentos.parquet"
hist = pd.read_parquet(hist_path, columns=["VERSION_ID", "CURRENT_FLAG"])
print(hist.groupby(["VERSION_ID", "CURRENT_FLAG"], dropna=False).size().rename("rows"))
assert (hist["VERSION_ID"] == 1).any(), "VERSION_ID=1 ausente no hist"
assert (hist["VERSION_ID"] == 2).any(), "VERSION_ID=2 ausente no hist"
print("OK — hist com v1 + v2 (e outras, se houver)")


In [65]:
hist_silver_dedup.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 809971 entries, 0 to 809970
Data columns (total 71 columns):
 #   Column                                                 Non-Null Count   Dtype         
---  ------                                                 --------------   -----         
 0   IDENTIFICACAO_NOTIFICACAO                              809971 non-null  object        
 1   RELACAO_MEDICAMENTO_EVENTO                             808372 non-null  object        
 2   NOME_MEDICAMENTO_WHODRUG                               804658 non-null  object        
 3   PRINCIPIOS_ATIVOS_WHODRUG                              803661 non-null  object        
 4   ATC_CODE_LEVEL_4                                       804658 non-null  object        
 5   DETENTOR_REGISTRO                                      138813 non-null  object        
 6   CONCENTRACAO                                           142948 non-null  object        
 7   COMPONENTE_SUSPEITO                                    8

# 🥇 Gold

In [66]:
hist_silver_dedup.columns

Index(['IDENTIFICACAO_NOTIFICACAO', 'RELACAO_MEDICAMENTO_EVENTO',
       'NOME_MEDICAMENTO_WHODRUG', 'PRINCIPIOS_ATIVOS_WHODRUG',
       'ATC_CODE_LEVEL_4', 'DETENTOR_REGISTRO', 'CONCENTRACAO',
       'COMPONENTE_SUSPEITO', 'ACAO_ADOTADA',
       'PROBLEMAS_ADICIONAIS_RELCIONADOS_MEDICAMENTO', 'INDICACAO_MEDDRA',
       'INDICACAO_RELATADA_NOTIFICADOR_INICIAL', 'DOSE', 'FREQUENCIA_DOSE',
       'POSOLOGIA', 'DURACAO', 'INICIO_ADMINISTRACAO', 'FIM_ADMINISTRACAO',
       'FORMA_FARMACEUTICA', 'VIA_ADMINISTRACAO', 'VIA_ADMINISTRACAO_MAE_PAI',
       'NUMELO_LOTE', 'ATUALIZADO', 'HASH_BRONZE',
       'RELACAO_MEDICAMENTO_EVENTO_VALOR', 'RELACAO_MEDICAMENTO_EVENTO_CHAVE',
       'COMPONENTE_SUSPEITO_VALOR', 'COMPONENTE_SUSPEITO_CHAVE',
       'ACAO_ADOTADA_VALOR', 'ACAO_ADOTADA_CHAVE', 'DURACAO_TIPO_VALOR',
       'DURACAO_TIPO_CHAVE', 'DURACAO_VALOR', 'CONCENTRACAO_TIPO_VALOR',
       'CONCENTRACAO_TIPO_CHAVE', 'CONCENTRACAO_VALOR',
       'CONCENTRACAO_VALOR_NUMERADOR', 'CONCENTRACAO_VALO

In [ ]:
# GOLD: apenas a versão atual da silver (ATC_LEVEL_5 já veio da silver)
hist_path = Path(project_root) / "data/02_silver/hist_fat_medicamentos/hist_fat_medicamentos.parquet"
silver_current = load_current_version(hist_path)
_vid = silver_current["VERSION_ID"].iloc[0]
print(f"Gold input (CURRENT_FLAG): {len(silver_current):,} | VERSION_ID={_vid}")

gold_cols = [
    "IDENTIFICACAO_NOTIFICACAO",
    "RELACAO_MEDICAMENTO_EVENTO_CHAVE",
    "RELACAO_MEDICAMENTO_EVENTO_VALOR",
    "NOME_MEDICAMENTO_WHODRUG",
    "PRINCIPIOS_ATIVOS_WHODRUG",
    "ATC_CODE_LEVEL_4",
    "ATC_CODE_LEVEL_5",
    "ATC_LEVEL_5",
    "ATC_LEVEL_5_SCORE",
    "DETENTOR_REGISTRO",
    "DETENTOR_REGISTRO_SCORE",
    "DETENTOR_REGISTRO_CHAVE",
    "CONCENTRACAO_TIPO_CHAVE",
    "CONCENTRACAO_TIPO_VALOR",
    "CONCENTRACAO_VALOR",
    "CONCENTRACAO_VALOR_NUMERADOR",
    "CONCENTRACAO_VALOR_DENOMINADOR",
    "COMPONENTE_SUSPEITO_CHAVE",
    "COMPONENTE_SUSPEITO_VALOR",
    "ACAO_ADOTADA_CHAVE",
    "ACAO_ADOTADA_VALOR",
    "PROB_ADIC_ABUSO",
    "PROB_ADIC_ERRO_DE_MEDICACAO",
    "PROB_ADIC_EXPOSICAO_OCUPACIONAL",
    "PROB_ADIC_FALSIFICACAO",
    "PROB_ADIC_LOTES_TESTADOS_E_DENTRO_DAS_ESPECIFICACOES",
    "PROB_ADIC_LOTES_TESTADOS_E_FORA_DAS_ESPECIFICACOES",
    "PROB_ADIC_MEDICAMENTO_TOMADO_FORA_DA_DATA_DE_VALIDADE",
    "PROB_ADIC_MEDICAMENTO_TOMADO_PELO_PAI",
    "PROB_ADIC_SUPERDOSE",
    "PROB_ADIC_USO_INCORRETO",
    "PROB_ADIC_USO_OFF_LABEL_USO_SEM_REGISTRO",
    "REACAO_CHAVE_x",
    "REACAO_CHAVE_y",
    "DOSE_TIPO_CHAVE",
    "DOSE_TIPO_VALOR",
    "DOSE_VALOR",
    "FREQUENCIA_DOSE",
    "POSOLOGIA",
    "DURACAO_TIPO_CHAVE",
    "DURACAO_TIPO_VALOR",
    "DURACAO_VALOR",
    "INICIO_ADMINISTRACAO",
    "FIM_ADMINISTRACAO",
    "FORMA_FARMACEUTICA_CHAVE",
    "VIA_ADMINISTRACAO_CHAVE",
    "VIA_ADMINISTRACAO_MAE_PAI_CHAVE",
    "NUMELO_LOTE",
    "ATUALIZADO",
    "HASH_BRONZE",
    "HASH_SILVER",
    "VERSION_ID",
    "CURRENT_FLAG",
    "DATE_UPDATE",
]

rename_map = {
    "REACAO_CHAVE_x": "INDICACAO_MEDDRA_REACAO_CHAVE",
    "REACAO_CHAVE_y": "INDICACAO_RELATADA_NOTIFICADOR_INICIAL_REACAO_CHAVE",
}

final_cols = [rename_map.get(col, col) for col in gold_cols]
fat = silver_current.rename(columns=rename_map)
fat = fat[final_cols]


In [68]:
gold_hash_exclude = ["ATUALIZADO", "HASH_BRONZE", "HASH_SILVER", "VERSION_ID", "CURRENT_FLAG", "DATE_UPDATE", "ATC_LEVEL_5_SCORE"]
gold_hash_cols = [c for c in final_cols if c not in gold_hash_exclude]
fat["HASH_GOLD"] = build_row_hash(fat, gold_hash_cols, algo="sha256", sep="|")


In [69]:
fat.head()

,IDENTIFICACAO_NOTIFICACAO,RELACAO_MEDICAMENTO_EVENTO_CHAVE,RELACAO_MEDICAMENTO_EVENTO_VALOR,NOME_MEDICAMENTO_WHODRUG,PRINCIPIOS_ATIVOS_WHODRUG,ATC_CODE_LEVEL_4,ATC_CODE_LEVEL_5,ATC_LEVEL_5,ATC_LEVEL_5_SCORE,DETENTOR_REGISTRO,DETENTOR_REGISTRO_SCORE,DETENTOR_REGISTRO_CHAVE,CONCENTRACAO_TIPO_CHAVE,CONCENTRACAO_TIPO_VALOR,CONCENTRACAO_VALOR,CONCENTRACAO_VALOR_NUMERADOR,CONCENTRACAO_VALOR_DENOMINADOR,COMPONENTE_SUSPEITO_CHAVE,COMPONENTE_SUSPEITO_VALOR,ACAO_ADOTADA_CHAVE,ACAO_ADOTADA_VALOR,PROB_ADIC_ABUSO,PROB_ADIC_ERRO_DE_MEDICACAO,PROB_ADIC_EXPOSICAO_OCUPACIONAL,PROB_ADIC_FALSIFICACAO,PROB_ADIC_LOTES_TESTADOS_E_DENTRO_DAS_ESPECIFICACOES,PROB_ADIC_LOTES_TESTADOS_E_FORA_DAS_ESPECIFICACOES,PROB_ADIC_MEDICAMENTO_TOMADO_FORA_DA_DATA_DE_VALIDADE,PROB_ADIC_MEDICAMENTO_TOMADO_PELO_PAI,PROB_ADIC_SUPERDOSE,PROB_ADIC_USO_INCORRETO,PROB_ADIC_USO_OFF_LABEL_USO_SEM_REGISTRO,INDICACAO_MEDDRA_REACAO_CHAVE,INDICACAO_RELATADA_NOTIFICADOR_INICIAL_REACAO_CHAVE,DOSE_TIPO_CHAVE,DOSE_TIPO_VALOR,DOSE_VALOR,FREQUENCIA_DOSE,POSOLOGIA,DURACAO_TIPO_CHAVE,DURACAO_TIPO_VALOR,DURACAO_VALOR,INICIO_ADMINISTRACAO,FIM_ADMINISTRACAO,FORMA_FARMACEUTICA_CHAVE,VIA_ADMINISTRACAO_CHAVE,VIA_ADMINISTRACAO_MAE_PAI_CHAVE,NUMELO_LOTE,ATUALIZADO,HASH_BRONZE,HASH_SILVER,VERSION_ID,CURRENT_FLAG,DATE_UPDATE,HASH_GOLD
0,BR-ANVISA-300000004,1,SUSPEITO,OXACILINA,OXACILLIN SODIUM,J01CF,J01CF04,J01CF04_Oxacillin,100.0,None,0.0,0,0,DESCONHECIDO,NaN,NaN,NaN,0,DESCONHECIDO,0,DESCONHECIDO,0,0,0,0,0,0,0,0,0,0,0,0,0,1,MILLIGRAM (MG),250.0,6 HORA,None,4,DIA,4.0,2018-11-24,NaT,0,6,5,5833018,2026-01-15,3E94C6FE0A6106B170D0EA675DCBCDDEAE7B2A96987C79E898FDB29A3C68E0CC,e7d2562343920063f3432065ddf46c988204b02bdf44d9ee9f82c2d32007ef51,2,True,2026-08-07,435c205975dfd3e0176a420dd74cfbcd9f18593fd39fe13152bf0298820ad6db
1,BR-ANVISA-300000005,1,SUSPEITO,PARACEMOL,PARACETAMOL,N02BE,N02BE01,N02BE01_Paracetamol,100.0,None,0.0,0,0,DESCONHECIDO,NaN,NaN,NaN,0,DESCONHECIDO,1,RETIRADA DO MEDICAMENTO,0,0,0,0,0,0,0,0,0,0,0,0,0,0,DESCONHECIDO,NaN,None,None,0,DESCONHECIDO,NaN,2018-11-22,2018-11-22,0,5,5,None,2026-01-15,845CF3FFA604507A5F50CBFF0488491FF396AE79C33B0D22D9C495A5CE3E3274,89777b97c47218b7f7b26c6bed10449a523a70b69d0acae41128f0101330bcb3,2,True,2026-08-07,7af038ce68e52252efcad89799a9dea65585aa9387a5dfb31005c932dbfd8f46
2,BR-ANVISA-300000007,1,SUSPEITO,HIDANTAL,PHENYTOIN,N03AB,N03AB02,N03AB02_Phenytoin,100.0,None,0.0,0,0,DESCONHECIDO,NaN,NaN,NaN,0,DESCONHECIDO,1,RETIRADA DO MEDICAMENTO,0,0,0,0,0,0,0,0,0,0,0,0,0,0,DESCONHECIDO,NaN,None,100MG A CADA 8 HORAS,0,DESCONHECIDO,NaN,2018-10-16,2018-11-15,0,5,5,None,2026-01-15,6C158BCB92540E9E2F9E8EAAD7CD5D3291FC69F1117E52EEBEF1678BCB0E0810,f7635b44704e87aff94c65fe9e710a0c25854847ae2c9f4a3da1c9409283c4b0,2,True,2026-08-07,4c35e6dc14df0f60d6ce4020378d347191636335954180603c135b3a7fa937e5
3,BR-ANVISA-300000010,1,SUSPEITO,MACRODANTINA,NITROFURANTOIN,J01XE,J01XE01,J01XE01_Nitrofurantoin,100.0,None,0.0,0,0,DESCONHECIDO,NaN,NaN,NaN,0,DESCONHECIDO,0,DESCONHECIDO,0,0,0,0,0,0,0,0,0,0,0,0,0,0,DESCONHECIDO,NaN,None,None,0,DESCONHECIDO,NaN,NaT,NaT,0,5,5,None,2026-01-15,CFA3E0EF0E8E2F45F5BBDDBE5A0E86C8311DCCC0AD52549747E075A9D2A24484,2174a0f1f6f99287383214792d505e581a423f0b1f9cbffb1cec3084a8e9bc12,2,True,2026-08-07,32a181f18f7f1e68bae059e35717f82730a60ac0c90322907f850d3411e28ecb
4,BR-ANVISA-300000011,1,SUSPEITO,DEXA-CITONEURIN,CYANOCOBALAMIN|DEXAMETHASONE|PYRIDOXINE HYDROCHLORIDE|THIAMINE HYDROCHLORIDE,H02BX,B03BA01,B03BA01_Cyanocobalamin,100.0,MERCK,100.0,202,0,DESCONHECIDO,NaN,NaN,NaN,0,DESCONHECIDO,1,RETIRADA DO MEDICAMENTO,0,0,0,0,0,0,0,0,0,0,0,0,0,0,DESCONHECIDO,NaN,None,None,4,DIA,2.0,2018-11-26,NaT,0,6,5,None,2026-01-15,AD468B6EA159A9618F5FF96295E52163AB9B4652312300E61D8DFC0A176C1E8B,57277c3804647a76647563627192689afbd7cb87d4a878ac30bedb6e79151c56,2,True,2026-08-07,79b611a9f81c3fbddabb2949c6aa28684db0012e6e9d2fb46dbfa955b3f2c205


### Check desagregação ATC (casos-guia)

- `BR-ANVISA-300331894`: 1 ATC + vários APIs → **1 linha**
- `BR-ANVISA-300294241`: 3 ATCs ≠ 2 APIs → **3 linhas** (só ATC explode; princípios completos em cada linha) + `ATC_LEVEL_5` por match
- `BR-ANVISA-300160134`: conferência adicional


In [70]:
ids_check = [
    'BR-ANVISA-300160134',
    'BR-ANVISA-300331894',
    'BR-ANVISA-300294241',
]
cols_check = [
    'IDENTIFICACAO_NOTIFICACAO',
    'NOME_MEDICAMENTO_WHODRUG',
    'PRINCIPIOS_ATIVOS_WHODRUG',
    'ATC_CODE_LEVEL_4',
    'ATC_CODE_LEVEL_5',
    'ATC_LEVEL_5',
    'ATC_LEVEL_5_SCORE',
]

for _id in ids_check:
    subset = fat.loc[fat['IDENTIFICACAO_NOTIFICACAO'] == _id, cols_check]
    print('=' * 80)
    print(f'{_id}  →  {len(subset)} linha(s) na gold')
    display(subset)


BR-ANVISA-300160134  →  4 linha(s) na gold


,IDENTIFICACAO_NOTIFICACAO,NOME_MEDICAMENTO_WHODRUG,PRINCIPIOS_ATIVOS_WHODRUG,ATC_CODE_LEVEL_4,ATC_CODE_LEVEL_5,ATC_LEVEL_5,ATC_LEVEL_5_SCORE
227618,BR-ANVISA-300160134,ACIDO FOLINICO,CALCIUM FOLINATE,V03AF,V03AF03,V03AF03_Calcium Folinate,100.0
227619,BR-ANVISA-300160134,L CARTIN,LEVOCARNITINE HYDROCHLORIDE,A16AA,A16AA01,A16AA01_Levocarnitine,100.0
227620,BR-ANVISA-300160134,BETAINA,BETAINE CITRATE,A09AB,A09AB02,A09AB02_Betaine,100.0
227621,BR-ANVISA-300160134,DEXALGEN,DEXAMETHASONE SODIUM PHOSPHATE|HYDROXOCOBALAMIN HYDROCHLORIDE|METAMIZOLE SODIUM,H02BX,B03BA03,B03BA03_Hydroxocobalamin,100.0


BR-ANVISA-300331894  →  1 linha(s) na gold


,IDENTIFICACAO_NOTIFICACAO,NOME_MEDICAMENTO_WHODRUG,PRINCIPIOS_ATIVOS_WHODRUG,ATC_CODE_LEVEL_4,ATC_CODE_LEVEL_5,ATC_LEVEL_5,ATC_LEVEL_5_SCORE
440553,BR-ANVISA-300331894,BUSCOPAN COMPOSTO,BUTILBROMETO DE ESCOPOLAMINA_X000D_|DIPIRONA,A03DB,A03DB04,A03DB04_Butylscopolamine And Analgesics,100.0


BR-ANVISA-300294241  →  3 linha(s) na gold


,IDENTIFICACAO_NOTIFICACAO,NOME_MEDICAMENTO_WHODRUG,PRINCIPIOS_ATIVOS_WHODRUG,ATC_CODE_LEVEL_4,ATC_CODE_LEVEL_5,ATC_LEVEL_5,ATC_LEVEL_5_SCORE
731376,BR-ANVISA-300294241,"BUTYLSCOPOLAMINE BROMIDE,METAMIZOLE SODIUM",BUTILBROMETO DE ESCOPOLAMINA_X000D_|DIPIRONA,A03DB,A03DB04,A03DB04_Butylscopolamine And Analgesics,100.0
731377,BR-ANVISA-300294241,"BUTYLSCOPOLAMINE BROMIDE,METAMIZOLE SODIUM",BUTILBROMETO DE ESCOPOLAMINA_X000D_|DIPIRONA,A04AD,A04AD01,A04AD01_Scopolamine,100.0
731378,BR-ANVISA-300294241,"BUTYLSCOPOLAMINE BROMIDE,METAMIZOLE SODIUM",BUTILBROMETO DE ESCOPOLAMINA_X000D_|DIPIRONA,N02BB,N02BB02,N02BB02_Metamizole,100.0


In [71]:
fat.to_parquet(Path(project_root) / "data/03_gold/fat_medicamentos/fat_medicamentos.parquet", index=False)